# C-GASTON UNI Backbone

Utilize the UNI pretrained vision model for histology images instead of general image model ResNet-50.

In [1]:
# Run this cell once per Colab session (skip if already installed)
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('torch', 'torchvision')
pip('anndata', 'scanpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn', 'kneed')
pip('gaston-spatial')

print('Done.')

Done.


## Step 1: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

Mounted at /content/drive
Drive mounted at /content/drive


## Step 1.5: Import Packages

In [3]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
import scipy.sparse as sp
import anndata as ad
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from gaston import neural_net, dp_related
from huggingface_hub import login
import warnings
warnings.filterwarnings('ignore')

## Step 3: Configuration

In [4]:
BASE_DIR = "/content/drive/MyDrive/EECS 545 Project"
GLMPCA_UNIFIED = "/content/drive/MyDrive/EECS545_glmpca_results"
HE_BASE_DIR = f"{BASE_DIR}/DLPFC_Datasets(raw data)"
SAMPLE1_DATA_DIR = f"{HE_BASE_DIR}/Sample1/h5ad_cordintae_data"
SAMPLE3_DATA_DIR = f"{HE_BASE_DIR}/Sample3/h5ad_cordinate_data"
OUTPUT_DIR = "/content/drive/MyDrive/C_GASTON_results/standard"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
ALL_SLICES = ['151507', '151508', '151509', '151510',
              '151673', '151674', '151675', '151676']

SLICE_DATA_DIRS = {
    '151507': SAMPLE1_DATA_DIR,
    '151508': SAMPLE1_DATA_DIR,
    '151509': SAMPLE1_DATA_DIR,
    '151510': SAMPLE1_DATA_DIR,
    '151673': SAMPLE3_DATA_DIR,
    '151674': SAMPLE3_DATA_DIR,
    '151675': SAMPLE3_DATA_DIR,
    '151676': SAMPLE3_DATA_DIR,
}

HE_IMAGE_PATHS = {
    '151507': f'{HE_BASE_DIR}/Sample1/H&E image/151507',
    '151508': f'{HE_BASE_DIR}/Sample1/H&E image/151508',
    '151509': f'{HE_BASE_DIR}/Sample1/H&E image/151509',
    '151510': f'{HE_BASE_DIR}/Sample1/H&E image/151510',
    '151673': f'{HE_BASE_DIR}/Sample3/H&E image/151673',
    '151674': f'{HE_BASE_DIR}/Sample3/H&E image/151674',
    '151675': f'{HE_BASE_DIR}/Sample3/H&E image/151675',
    '151676': f'{HE_BASE_DIR}/Sample3/H&E image/151676',
}

LABEL_TO_INT = {'L1': 0, 'L2': 1, 'L3': 2, 'L4': 3, 'L5': 4, 'L6': 5, 'WM': 6}

## Step 4: Hyperparameters

In [6]:
# Vision Model Parameters
VISION_MODEL = "uni"
PATCH_SIZE = 224

# C-GASTON Specific
ISODEPTH_ARCH = [20, 20]
EXPRESSION_ARCH = [20, 20]
NUM_DIMS = 14

NUM_LAYERS = 7
EMBEDDING_DIM = 128
TEMPERATURE = 0.07
LAMBDA_CONTRASTIVE = 0.1
SIGMA_SOFT = 0.5

# Training Parameters
BATCH_SIZE = 256
WARMUP_EPOCHS = 2000
TOTAL_EPOCHS = 10000
NUM_RESTARTS = 10
LR = 1e-3

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Configuration loaded.")
print(f"Slices: {ALL_SLICES}")
print(f"Vision backbone: {VISION_MODEL}")
print(f"λ={LAMBDA_CONTRASTIVE}, τ={TEMPERATURE}, D={EMBEDDING_DIM}")
print(f"Device: {device}")
print(f"Training: {WARMUP_EPOCHS} warmup + {TOTAL_EPOCHS - WARMUP_EPOCHS} joint epochs")
print(f"Restarts: {NUM_RESTARTS}")

Configuration loaded.
Slices: ['151507', '151508', '151509', '151510', '151673', '151674', '151675', '151676']
Vision backbone: uni
λ=0.1, τ=0.07, D=128
Device: cuda
Training: 2000 warmup + 8000 joint epochs
Restarts: 10


## Step 5: Helper Functions

In [7]:
# ============================================================
# z-score normalization (same as gaston's load_rescale_input_data)
# ============================================================
def load_rescale_input_data(S, A):
    S_mean, S_std = S.mean(axis=0), S.std(axis=0)
    A_mean, A_std = A.mean(axis=0), A.std(axis=0)
    S_norm = (S - S_mean) / (S_std + 1e-8)
    A_norm = (A - A_mean) / (A_std + 1e-8)
    return torch.tensor(S_norm, dtype=torch.float32), torch.tensor(A_norm, dtype=torch.float32)

# ============================================================
# Loss functions
# ============================================================
def info_nce_loss(z_m, z_v, temperature=0.07):
    z_m = F.normalize(z_m, dim=1)
    z_v = F.normalize(z_v, dim=1)
    logits = z_m @ z_v.T / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_m2v = F.cross_entropy(logits, labels)
    loss_v2m = F.cross_entropy(logits.T, labels)
    return (loss_m2v + loss_v2m) / 2

def soft_weighted_info_nce_loss(z_m, z_v, isodepth, temperature=0.07, sigma=0.5):
    """
    Soft-Weighted InfoNCE

    Weights negatives by isodepth distance using an RBF kernel:
        w_ij = 1 - exp(-(d_i - d_j)^2 / σ^2)

    Spots with similar isodepth (same layer) have w_ij ≈ 0 (suppressed negatives).
    Spots with different isodepth have w_ij ≈ 1 (kept as negatives).

    Parameters
    ----------
    z_m      : (B, D) molecular embeddings
    z_v      : (B, D) vision embeddings
    isodepth : (B, 1) or (B,) scalar isodepth values
    temperature : float, τ
    sigma    : float, σ (RBF bandwidth)

    Returns
    -------
    loss : scalar
    """
    z_m = F.normalize(z_m, dim=1)
    z_v = F.normalize(z_v, dim=1)

    # Isodepth distance matrix
    d = isodepth.view(-1, 1)  # (B, 1)
    dist_sq = (d - d.T) ** 2   # (B, B)

    # RBF weights: w_ij = 1 - exp(-dist^2 / σ^2)
    weights = 1.0 - torch.exp(-dist_sq / (sigma ** 2))  # (B, B)

    # Cosine similarity / temperature
    logits = z_m @ z_v.T / temperature  # (B, B)

    # Weighted softmax (numerator is the positive pair = diagonal)
    B = logits.shape[0]

    # For the denominator, weight each term by w_ij
    # But keep the positive pair (diagonal) weight = 1
    weights_with_pos = weights.clone()
    weights_with_pos[torch.arange(B), torch.arange(B)] = 1.0

    # Numerator: exp(sim(z_m_i, z_v_i) / τ)
    pos_logits = torch.diag(logits)  # (B,)

    # Denominator: sum_j w_ij * exp(sim(z_m_i, z_v_j) / τ)
    weighted_exp = weights_with_pos * torch.exp(logits)  # (B, B)
    denom = weighted_exp.sum(dim=1)  # (B,)

    # Loss: -1/B * sum_i log(exp(pos) / denom)
    loss = -torch.mean(pos_logits - torch.log(denom + 1e-8))

    return loss

# ============================================================
# H&E patch extraction
# ============================================================
def extract_patches(slice_id, adata, he_dir, output_size=224, crop_multiplier=3.0):
    img = Image.open(f'{he_dir}/tissue_hires_image.png').convert('RGB')
    img_np = np.array(img)
    H_img, W_img = img_np.shape[:2]

    with open(f'{he_dir}/scalefactors_json.json') as f:
        sf = json.load(f)
    scale = sf['tissue_hires_scalef']
    spot_d_fullres = sf['spot_diameter_fullres']
    spot_d_hires = spot_d_fullres * scale
    crop_radius = int(np.ceil(spot_d_hires * crop_multiplier / 2))

    print(f"  Hires image: {W_img}x{H_img}, scale={scale:.4f}")
    print(f"  Spot diameter (hires): {spot_d_hires:.1f} px")
    print(f"  Crop size: {2*crop_radius}x{2*crop_radius} -> {output_size}x{output_size}")

    pos_df = {}
    with open(f'{he_dir}/tissue_positions_list.txt') as f:
        for line in f:
            parts = line.strip().split(',')
            pos_df[parts[0]] = (float(parts[4]), float(parts[5]))

    barcodes = adata.obs_names.tolist()
    patches = np.zeros((len(barcodes), output_size, output_size, 3), dtype=np.uint8)

    for i, bc in enumerate(barcodes):
        if bc not in pos_df:
            continue
        row_fullres, col_fullres = pos_df[bc]
        row_hires = row_fullres * scale
        col_hires = col_fullres * scale
        r1 = max(0, int(row_hires - crop_radius))
        r2 = min(H_img, int(row_hires + crop_radius))
        c1 = max(0, int(col_hires - crop_radius))
        c2 = min(W_img, int(col_hires + crop_radius))
        crop = img_np[r1:r2, c1:c2]
        if crop.shape[0] < 2 or crop.shape[1] < 2:
            continue
        patches[i] = np.array(Image.fromarray(crop).resize((output_size, output_size), Image.LANCZOS))

    return patches

# ============================================================
# Vision feature extraction
# ============================================================
patch_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_vision_features(patches, model, D_v, batch_size=64):
    model = model.to(device)
    N = len(patches)
    features = np.zeros((N, D_v), dtype=np.float32)
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        batch_tensors = torch.stack([
            patch_transform(Image.fromarray(p)) for p in patches[start:end]
        ]).to(device)

        with torch.no_grad():
            feats = model(batch_tensors)

        features[start:end] = feats.cpu().numpy()

        if (start // batch_size) % 10 == 0:
            print(f"  Processed {end}/{N} patches...", end='\r')

    print(f"  Processed {N}/{N} patches.     ")
    return features

# ============================================================
# Vision model selection
# ============================================================
def load_vision_model(name):
      if name == "resnet":
          import torchvision.models as models
          model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
          dim = 2048

      elif name == "uni":
          import timm
          login()
          model = timm.create_model(
              "hf-hub:MahmoodLab/UNI",
              pretrained=True,
              init_values=1e-5,
              dynamic_img_size=True
          )
          dim = 1024

      elif name == "conch":
          from conch.open_clip_custom import create_model_from_pretrained
          model, _ = create_model_from_pretrained(
              "conch_ViT-B-16", "hf_hub:MahmoodLab/conch"
          )
          # wrap to return image features only
          class CONCHVisionWrapper(nn.Module):
              def __init__(self, m): super().__init__(); self.m = m
              def forward(self, x): return self.m.encode_image(x, proj_contrast=False, normalize=False)
          model = CONCHVisionWrapper(model)
          dim = 512

      else:
          raise ValueError(f"Unknown vision model: {name}")

      model.eval()
      for param in model.parameters():
          param.requires_grad = False
      return model, dim

## Step 6: CGASTON Model

In [8]:
# ============================================================
# C-GASTON Model
# ============================================================
class CGASTON(nn.Module):
    def __init__(self, K, D_v, D=128, isodepth_arch=[20, 20], expression_arch=[20, 20]):
        super().__init__()
        self.K = K
        self.D = D

        enc_layers = []
        dims = [2] + isodepth_arch + [1]
        for i in range(len(dims) - 1):
            enc_layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                enc_layers.append(nn.ReLU())
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        dims = [1] + expression_arch + [K]
        for i in range(len(dims) - 1):
            dec_layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                dec_layers.append(nn.ReLU())
        self.decoder = nn.Sequential(*dec_layers)

        self.mol_projection = nn.Linear(1 + K, D)
        self.vis_projection = nn.Sequential(
            nn.Linear(D_v, 512), nn.ReLU(), nn.LayerNorm(512), nn.Linear(512, D),
        )

    def encode_isodepth(self, S):
        return self.encoder(S)

    def decode_expression(self, d):
        return self.decoder(d)

    def molecular_embedding(self, S):
        d = self.encode_isodepth(S)
        z_hat = self.decode_expression(d)
        concat = torch.cat([d, z_hat], dim=1)
        z_m = self.mol_projection(concat)
        return z_m, d, z_hat

    def vision_embedding(self, v):
        return self.vis_projection(v)

    def forward(self, S, v):
        z_m, d, z_hat = self.molecular_embedding(S)
        z_v = self.vision_embedding(v)
        return z_m, z_v, d, z_hat

# ============================================================
# CGASTONWrapper for dp_related compatibility
# ============================================================
class CGASTONWrapper:
    def __init__(self, cgaston_model):
        self.spatial_embedding = cgaston_model.encoder
        self.expression_function = cgaston_model.decoder

## Step 7: Training Loop

In [9]:
# ============================================================
# Training loop
# ============================================================
def train_cgaston(
    model,
    S_torch,
    A_torch,
    V_torch,
    total_epochs=10000,
    warmup_epochs=2000,
    lam=0.1,
    temperature=0.07,
    sigma=0.5,
    batch_size=256,
    lr=1e-3,
    use_soft_weighting=False,
    log_interval=2000,
    seed=0,
):
    torch.manual_seed(seed)
    N = S_torch.shape[0]
    S = S_torch.to(device)
    A = A_torch.to(device)
    V = V_torch.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_fn = nn.MSELoss()
    history = {"recon": [], "contrastive": [], "total": []}

    for epoch in range(1, total_epochs + 1):
        model.train()
        z_m_full, d_full, z_hat_full = model.molecular_embedding(S)
        loss_recon = mse_fn(z_hat_full, A)

        loss_cont = torch.tensor(0.0, device=device)
        if epoch > warmup_epochs:
            perm = torch.randperm(N, device=device)[:batch_size]
            z_m_batch, d_batch, _ = model.molecular_embedding(S[perm])
            z_v_batch = model.vision_embedding(V[perm])
            if use_soft_weighting:
                loss_cont = soft_weighted_info_nce_loss(
                    z_m_batch, z_v_batch, d_batch.squeeze(), temperature, sigma
                )
            else:
                loss_cont = info_nce_loss(z_m_batch, z_v_batch, temperature)

        current_lam = lam if epoch > warmup_epochs else 0.0
        loss_total = loss_recon + current_lam * loss_cont

        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()

        history["recon"].append(loss_recon.item())
        history["contrastive"].append(loss_cont.item())
        history["total"].append(loss_total.item())

        if epoch % log_interval == 0:
            phase = "WARMUP" if epoch <= warmup_epochs else "JOINT "
            print(
                f"[{phase}] Epoch {epoch:5d}/{total_epochs} | "
                f"Recon: {loss_recon.item():.4f} | "
                f"Cont: {loss_cont.item():.4f} | "
                f"Total: {loss_total.item():.4f}"
            )

    return model, history

## Step 8: Pipeline

In [10]:
print("=" * 60)
print(f"C-GASTON Standard — {ALL_SLICES}")
print("=" * 60)

# --- Step 1: Load pre-computed GLM-PCA + adata ---
print("\n[Step 1] Loading pre-computed GLM-PCA...")
data = {}
for sid in ALL_SLICES:
    print(f"\n--- {sid} ---")

    # Load h5ad (gene expression + spatial coordinates + ground truth)
    adata = ad.read_h5ad(f"{SLICE_DATA_DIRS[sid]}/{sid}.h5ad")
    adata.var_names_make_unique()

    X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
    S = np.asarray(adata.obsm["spatial"])

    # Ground Truth Labels
    gt_str = adata.obs["original_domain"].astype(str).values
    gt_int = np.array([LABEL_TO_INT.get(label, -1) for label in gt_str])

    # Load pre-computed GLM-PCA factors (K=14) from GASTON pipeline
    A = np.load(f"{GLMPCA_UNIFIED}/{sid}/glmpca.npy")
    print(f"  Loaded GLM-PCA: A {A.shape} from {GLMPCA_UNIFIED}/{sid}/")

    S_torch, A_torch = load_rescale_input_data(S, A)
    data[sid] = {
        "adata": adata,
        "coords": S,
        "gt_labels": gt_int,
        "A_torch": A_torch,
        "S_torch": S_torch,
    }
    print(f"  S_torch: {S_torch.shape}, A_torch: {A_torch.shape}")

C-GASTON Standard — ['151507', '151508', '151509', '151510', '151673', '151674', '151675', '151676']

[Step 1] Loading pre-computed GLM-PCA...

--- 151507 ---
  Loaded GLM-PCA: A (4221, 14) from /content/drive/MyDrive/EECS545_glmpca_results/151507/
  S_torch: torch.Size([4221, 2]), A_torch: torch.Size([4221, 14])

--- 151508 ---
  Loaded GLM-PCA: A (4381, 14) from /content/drive/MyDrive/EECS545_glmpca_results/151508/
  S_torch: torch.Size([4381, 2]), A_torch: torch.Size([4381, 14])

--- 151509 ---
  Loaded GLM-PCA: A (4788, 14) from /content/drive/MyDrive/EECS545_glmpca_results/151509/
  S_torch: torch.Size([4788, 2]), A_torch: torch.Size([4788, 14])

--- 151510 ---
  Loaded GLM-PCA: A (4595, 14) from /content/drive/MyDrive/EECS545_glmpca_results/151510/
  S_torch: torch.Size([4595, 2]), A_torch: torch.Size([4595, 14])

--- 151673 ---
  Loaded GLM-PCA: A (3611, 14) from /content/drive/MyDrive/EECS545_glmpca_results/151673/
  S_torch: torch.Size([3611, 2]), A_torch: torch.Size([3611, 14

In [11]:
# --- Step 2: H&E patches ---
print("\n[Step 2] Extracting H&E patches...")
for sid in ALL_SLICES:
    print(f"\n--- {sid} ---")
    patches = extract_patches(sid, data[sid]['adata'], HE_IMAGE_PATHS[sid],
                               output_size=PATCH_SIZE, crop_multiplier=3.0)
    data[sid]['patches'] = patches
    print(f"  Patches: {patches.shape}")


[Step 2] Extracting H&E patches...

--- 151507 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.5 px
  Crop size: 44x44 -> 224x224
  Patches: (4221, 224, 224, 3)

--- 151508 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.5 px
  Crop size: 44x44 -> 224x224
  Patches: (4381, 224, 224, 3)

--- 151509 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.6 px
  Crop size: 44x44 -> 224x224
  Patches: (4788, 224, 224, 3)

--- 151510 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.5 px
  Crop size: 44x44 -> 224x224
  Patches: (4595, 224, 224, 3)

--- 151673 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.5 px
  Crop size: 44x44 -> 224x224
  Patches: (3611, 224, 224, 3)

--- 151674 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter (hires): 14.5 px
  Crop size: 44x44 -> 224x224
  Patches: (3635, 224, 224, 3)

--- 151675 ---
  Hires image: 2000x2000, scale=0.1500
  Spot diameter

In [13]:
# --- Step 3: Vision features ---
print(f"\n[Step 3] Extracting vision features ({VISION_MODEL})...")
vision_model, D_v = load_vision_model(VISION_MODEL)
vision_model.fc = nn.Identity()

for sid in ALL_SLICES:
    print(f"\n--- {sid} ---")
    features = extract_vision_features(data[sid]['patches'], vision_model, D_v)
    data[sid]['vision_features'] = features
    del data[sid]['patches']
    print(f"  Vision features: {features.shape}")


[Step 3] Extracting vision features (uni)...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]


--- 151507 ---
  Processed 4221/4221 patches.     
  Vision features: (4221, 1024)

--- 151508 ---
  Processed 4381/4381 patches.     
  Vision features: (4381, 1024)

--- 151509 ---
  Processed 4788/4788 patches.     
  Vision features: (4788, 1024)

--- 151510 ---
  Processed 4595/4595 patches.     
  Vision features: (4595, 1024)

--- 151673 ---
  Processed 3611/3611 patches.     
  Vision features: (3611, 1024)

--- 151674 ---
  Processed 3635/3635 patches.     
  Vision features: (3635, 1024)

--- 151675 ---
  Processed 3566/3566 patches.     
  Vision features: (3566, 1024)

--- 151676 ---
  Processed 3431/3431 patches.     
  Vision features: (3431, 1024)


In [14]:
print("="*60)
print(f"C-GASTON Training — {ALL_SLICES}")
print("="*60)

all_results = {}
for sid in ALL_SLICES:
    save_dir = f'{OUTPUT_DIR}/{sid}'
    best_model_path = f'{save_dir}/cgaston_best.pt'

    if os.path.isfile(best_model_path):
        print(f'\n{sid}: already trained, skipping.')
        # Load saved model so eval cell can use it
        mdl = CGASTON(K=NUM_DIMS, D_v=D_v, D=EMBEDDING_DIM,
                      isodepth_arch=ISODEPTH_ARCH,
                      expression_arch=EXPRESSION_ARCH).to(device)
        mdl.load_state_dict(torch.load(best_model_path, map_location=device))
        mdl.eval()
        all_results[sid] = {'model': mdl, 'history': None, 'best_loss': float('nan')}
        continue

    print(f"\n{'='*60}")
    print(f"Training C-GASTON on slice {sid}")
    print(f"{'='*60}")

    S_t = data[sid]['S_torch']
    A_t = data[sid]['A_torch']
    V_t = torch.tensor(data[sid]['vision_features'], dtype=torch.float32)

    best_loss = float('inf')
    best_model_state = None
    best_history = None

    for restart in range(NUM_RESTARTS):
        print(f"\n--- Restart {restart}/{NUM_RESTARTS-1} ---")
        mdl = CGASTON(K=NUM_DIMS, D_v=D_v, D=EMBEDDING_DIM,
                      isodepth_arch=ISODEPTH_ARCH,
                      expression_arch=EXPRESSION_ARCH).to(device)

        mdl, hist = train_cgaston(
            mdl, S_t, A_t, V_t,
            total_epochs=TOTAL_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
            lam=LAMBDA_CONTRASTIVE, temperature=TEMPERATURE, sigma=SIGMA_SOFT,
            batch_size=BATCH_SIZE, lr=LR, log_interval=2000, seed=restart)

        final_recon = hist['recon'][-1]
        print(f"  Final recon loss: {final_recon:.6f}")
        if final_recon < best_loss:
            best_loss = final_recon
            best_model_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}
            best_history = hist
            print(f"  >>> New best! (loss={best_loss:.6f})")

    best_model = CGASTON(K=NUM_DIMS, D_v=D_v, D=EMBEDDING_DIM,
                         isodepth_arch=ISODEPTH_ARCH,
                         expression_arch=EXPRESSION_ARCH).to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()

    os.makedirs(save_dir, exist_ok=True)
    torch.save(best_model.state_dict(), best_model_path)

    all_results[sid] = {'model': best_model, 'history': best_history, 'best_loss': best_loss}
    print(f"\n{sid}: Best model saved (recon loss = {best_loss:.6f})")

C-GASTON Training — ['151507', '151508', '151509', '151510', '151673', '151674', '151675', '151676']

151507: already trained, skipping.

151508: already trained, skipping.

151509: already trained, skipping.

151510: already trained, skipping.

151673: already trained, skipping.

151674: already trained, skipping.

151675: already trained, skipping.

151676: already trained, skipping.


In [15]:
# --- Step 5: Evaluate ---
print("\n[Step 5] Evaluating...")
eval_results = {}
for sid in ALL_SLICES:
    print(f"\n--- Evaluating {sid} ---")
    mdl = all_results[sid]['model']
    mdl.eval()
    mdl_cpu = mdl.cpu()

    A_np = data[sid]['A_torch'].detach().numpy()
    S_np = data[sid]['S_torch'].detach().numpy()
    wrapper = CGASTONWrapper(mdl_cpu)

    gaston_isodepth, gaston_labels = dp_related.get_isodepth_labels(
        wrapper, A_np, S_np, NUM_LAYERS, num_buckets=100)

    gt = data[sid]['gt_labels']
    ari = adjusted_rand_score(gt, gaston_labels)
    nmi = normalized_mutual_info_score(gt, gaston_labels)
    d_norm = (gaston_isodepth - gaston_isodepth.min()) / (gaston_isodepth.max() - gaston_isodepth.min() + 1e-8)
    corr = abs(np.corrcoef(d_norm, gt)[0, 1])

    eval_results[sid] = {
        'isodepth': gaston_isodepth, 'labels': gaston_labels.astype(int),
        'ari': ari, 'nmi': nmi, 'corr': corr,
    }

    np.save(f'{OUTPUT_DIR}/{sid}/cgaston_isodepth.npy', gaston_isodepth)
    np.save(f'{OUTPUT_DIR}/{sid}/cgaston_labels.npy', gaston_labels)
    mdl.to(device)

    print(f"  ARI: {ari:.4f}, NMI: {nmi:.4f}, Corr: {corr:.4f}")


[Step 5] Evaluating...

--- Evaluating 151507 ---
  ARI: 0.4871, NMI: 0.6217, Corr: 0.8234

--- Evaluating 151508 ---
  ARI: 0.4296, NMI: 0.5665, Corr: 0.4887

--- Evaluating 151509 ---
  ARI: 0.4107, NMI: 0.6054, Corr: 0.5270

--- Evaluating 151510 ---
  ARI: 0.3924, NMI: 0.5903, Corr: 0.5988

--- Evaluating 151673 ---
  ARI: 0.6691, NMI: 0.7449, Corr: 0.9747

--- Evaluating 151674 ---
  ARI: 0.5042, NMI: 0.6145, Corr: 0.8967

--- Evaluating 151675 ---
  ARI: 0.3308, NMI: 0.5149, Corr: 0.3100

--- Evaluating 151676 ---
  ARI: 0.5288, NMI: 0.6590, Corr: 0.9619


In [16]:
# --- Summary ---
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
for sid in ALL_SLICES:
    r = eval_results[sid]
    print(f"  {sid}: ARI={r['ari']:.4f}  NMI={r['nmi']:.4f}  Corr={r['corr']:.4f}  ReconLoss={all_results[sid]['best_loss']:.4f}")
print(f"\nResults saved to: {OUTPUT_DIR}/")
print("Done!")


RESULTS SUMMARY
  151507: ARI=0.4871  NMI=0.6217  Corr=0.8234  ReconLoss=nan
  151508: ARI=0.4296  NMI=0.5665  Corr=0.4887  ReconLoss=nan
  151509: ARI=0.4107  NMI=0.6054  Corr=0.5270  ReconLoss=nan
  151510: ARI=0.3924  NMI=0.5903  Corr=0.5988  ReconLoss=nan
  151673: ARI=0.6691  NMI=0.7449  Corr=0.9747  ReconLoss=nan
  151674: ARI=0.5042  NMI=0.6145  Corr=0.8967  ReconLoss=nan
  151675: ARI=0.3308  NMI=0.5149  Corr=0.3100  ReconLoss=nan
  151676: ARI=0.5288  NMI=0.6590  Corr=0.9619  ReconLoss=nan

Results saved to: /content/drive/MyDrive/C_GASTON_results/standard/
Done!


## Soft-Weighted InfoNCE Training

Re-train C-GASTON with `use_soft_weighting=True`. The soft-weighted InfoNCE uses isodepth distance to suppress false negatives (same-layer spots), addressing the layer-ordering degradation observed with standard InfoNCE.

In [17]:
OUTPUT_DIR = "/content/drive/MyDrive/C_GASTON_results/soft"

In [18]:
print("=" * 60)
print(f"C-GASTON Soft-InfoNCE Training — {ALL_SLICES}")
print("=" * 60)

all_results = {}
for sid in ALL_SLICES:
    save_dir = f"{OUTPUT_DIR}/{sid}"
    best_model_path = f"{save_dir}/cgaston_best.pt"

    if os.path.isfile(best_model_path):
        print(f"\n{sid}: already trained, skipping.")
        # Load saved model so eval cell can use it
        mdl = CGASTON(
            K=NUM_DIMS,
            D_v=D_v,
            D=EMBEDDING_DIM,
            isodepth_arch=ISODEPTH_ARCH,
            expression_arch=EXPRESSION_ARCH,
        ).to(device)
        mdl.load_state_dict(torch.load(best_model_path, map_location=device))
        mdl.eval()
        all_results[sid] = {"model": mdl, "history": None, "best_loss": float("nan")}
        continue

    print(f"\n{'=' * 60}")
    print(f"Training C-GASTON on slice {sid}")
    print(f"{'=' * 60}")

    S_t = data[sid]["S_torch"]
    A_t = data[sid]["A_torch"]
    V_t = torch.tensor(data[sid]["vision_features"], dtype=torch.float32)

    best_loss = float("inf")
    best_model_state = None
    best_history = None

    for restart in range(NUM_RESTARTS):
        print(f"\n--- Restart {restart}/{NUM_RESTARTS - 1} ---")
        mdl = CGASTON(
            K=NUM_DIMS,
            D_v=D_v,
            D=EMBEDDING_DIM,
            isodepth_arch=ISODEPTH_ARCH,
            expression_arch=EXPRESSION_ARCH,
        ).to(device)

        mdl, hist = train_cgaston(
            mdl,
            S_t,
            A_t,
            V_t,
            total_epochs=TOTAL_EPOCHS,
            warmup_epochs=WARMUP_EPOCHS,
            lam=LAMBDA_CONTRASTIVE,
            temperature=TEMPERATURE,
            sigma=SIGMA_SOFT,
            batch_size=BATCH_SIZE,
            lr=LR,
            use_soft_weighting=True,
            log_interval=2000,
            seed=restart,
        )

        final_recon = hist["recon"][-1]
        print(f"  Final recon loss: {final_recon:.6f}")
        if final_recon < best_loss:
            best_loss = final_recon
            best_model_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}
            best_history = hist
            print(f"  >>> New best! (loss={best_loss:.6f})")

    best_model = CGASTON(
        K=NUM_DIMS,
        D_v=D_v,
        D=EMBEDDING_DIM,
        isodepth_arch=ISODEPTH_ARCH,
        expression_arch=EXPRESSION_ARCH,
    ).to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()

    os.makedirs(save_dir, exist_ok=True)
    torch.save(best_model.state_dict(), best_model_path)

    all_results[sid] = {
        "model": best_model,
        "history": best_history,
        "best_loss": best_loss,
    }
    print(f"\n{sid}: Best model saved (recon loss = {best_loss:.6f})")

C-GASTON Soft-InfoNCE Training — ['151507', '151508', '151509', '151510', '151673', '151674', '151675', '151676']

151507: already trained, skipping.

151508: already trained, skipping.

151509: already trained, skipping.

151510: already trained, skipping.

Training C-GASTON on slice 151673

--- Restart 0/9 ---
[WARMUP] Epoch  2000/10000 | Recon: 0.5948 | Cont: 0.0000 | Total: 0.5948
[JOINT ] Epoch  4000/10000 | Recon: 0.5923 | Cont: 0.0621 | Total: 0.5985
[JOINT ] Epoch  6000/10000 | Recon: 0.5890 | Cont: 0.0509 | Total: 0.5941
[JOINT ] Epoch  8000/10000 | Recon: 0.5869 | Cont: 0.0715 | Total: 0.5940
[JOINT ] Epoch 10000/10000 | Recon: 0.6711 | Cont: 2.0443 | Total: 0.8755
  Final recon loss: 0.671087
  >>> New best! (loss=0.671087)

--- Restart 1/9 ---
[WARMUP] Epoch  2000/10000 | Recon: 0.5959 | Cont: 0.0000 | Total: 0.5959
[JOINT ] Epoch  4000/10000 | Recon: 0.5908 | Cont: 0.0833 | Total: 0.5991
[JOINT ] Epoch  6000/10000 | Recon: 0.5840 | Cont: 0.0533 | Total: 0.5893
[JOINT ] Epo

In [19]:
# --- Step 5: Evaluate ---
print("\n[Step 5] Evaluating...")
eval_results = {}
for sid in ALL_SLICES:
    print(f"\n--- Evaluating soft-weighted {sid} ---")
    mdl = all_results[sid]['model']
    mdl.eval()
    mdl_cpu = mdl.cpu()

    A_np = data[sid]['A_torch'].detach().numpy()
    S_np = data[sid]['S_torch'].detach().numpy()
    wrapper = CGASTONWrapper(mdl_cpu)

    gaston_isodepth, gaston_labels = dp_related.get_isodepth_labels(
        wrapper, A_np, S_np, NUM_LAYERS, num_buckets=100)

    gt = data[sid]['gt_labels']
    ari = adjusted_rand_score(gt, gaston_labels)
    nmi = normalized_mutual_info_score(gt, gaston_labels)
    d_norm = (gaston_isodepth - gaston_isodepth.min()) / (gaston_isodepth.max() - gaston_isodepth.min() + 1e-8)
    corr = abs(np.corrcoef(d_norm, gt)[0, 1])

    eval_results[sid] = {
        'isodepth': gaston_isodepth, 'labels': gaston_labels.astype(int),
        'ari': ari, 'nmi': nmi, 'corr': corr,
    }

    np.save(f'{OUTPUT_DIR}/{sid}/cgaston_soft_isodepth.npy', gaston_isodepth)
    np.save(f'{OUTPUT_DIR}/{sid}/cgaston_soft_labels.npy', gaston_labels)
    mdl.to(device)

    print(f"  ARI: {ari:.4f}, NMI: {nmi:.4f}, Corr: {corr:.4f}")


[Step 5] Evaluating...

--- Evaluating soft-weighted 151507 ---
  ARI: 0.4886, NMI: 0.5977, Corr: 0.7180

--- Evaluating soft-weighted 151508 ---
  ARI: 0.4310, NMI: 0.5740, Corr: 0.2753

--- Evaluating soft-weighted 151509 ---
  ARI: 0.4047, NMI: 0.5743, Corr: 0.6460

--- Evaluating soft-weighted 151510 ---
  ARI: 0.3063, NMI: 0.5075, Corr: 0.8222

--- Evaluating soft-weighted 151673 ---
  ARI: 0.6904, NMI: 0.7494, Corr: 0.9726

--- Evaluating soft-weighted 151674 ---
  ARI: 0.4919, NMI: 0.6088, Corr: 0.9215

--- Evaluating soft-weighted 151675 ---
  ARI: 0.4140, NMI: 0.5577, Corr: 0.8472

--- Evaluating soft-weighted 151676 ---
  ARI: 0.5342, NMI: 0.6657, Corr: 0.9699


In [20]:
from google.colab import runtime
runtime.unassign()